# Perseptron v2 - 00 Customer-Level Fold Hazirligi

Bu notebook proposal uyumlu customer-level 5-fold split dosyasini uretir. Ayni musteri ayni fold icinde hem train hem validation tarafinda yer almaz.

Ciktilar:

- `reports/proposal_v2/proposal_v2_fold_splits.csv`
- `reports/proposal_v2/proposal_v2_fold_protocol_summary.json`

## Ortam ve output klasorleri

Bu hucre Kaggle icin yazilabilir output klasorlerini hazirlar ve onceki notebook outputlari `Add Data` ile eklendiyse bunlari geri yukler. Repo icindeki `.py` modulleri import edilmez.

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
REPORTS_DIR = WORK_DIR / 'reports' / 'proposal_v2'
MODELS_DIR = WORK_DIR / 'models' / 'proposal_v2'
GRADCAM_DIR = REPORTS_DIR / 'gradcam_examples'
for path in [REPORTS_DIR, REPORTS_DIR / 'folds', MODELS_DIR, GRADCAM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def restore_previous_outputs():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return
    for root in input_root.glob('*'):
        if not root.is_dir():
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = root / relative
            target = WORK_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print(f'Restored previous output: {source} -> {target}')

restore_previous_outputs()
print('WORK_DIR    =', WORK_DIR)
print('REPORTS_DIR =', REPORTS_DIR)
print('MODELS_DIR  =', MODELS_DIR)

## Ortak sabitler

Bu hucre pathleri, feature listelerini, model adlarini ve deney varsayilanlarini tanimlar.

In [ ]:
from dataclasses import dataclass

DATA_DIR = WORK_DIR / 'data'
REPORTS_DIR = WORK_DIR / 'reports' / 'proposal_v2'
MODELS_DIR = WORK_DIR / 'models' / 'proposal_v2'

DEFAULT_SEED = 42
DEFAULT_N_FOLDS = 5
DEFAULT_VALIDATION_DAYS = 7

CUSTOMER_NUMERIC_FEATURES = ['FN', 'Active', 'age']
ARTICLE_NUMERIC_FEATURES = [
    'product_code',
    'product_type_no',
    'graphical_appearance_no',
    'colour_group_code',
    'perceived_colour_value_id',
    'perceived_colour_master_id',
    'department_no',
    'index_group_no',
    'section_no',
    'garment_group_no',
]
VISUAL_NUMERIC_FEATURES = ['visual_similarity', 'visual_history_count']
TABULAR_NUMERIC_FEATURES = CUSTOMER_NUMERIC_FEATURES + ARTICLE_NUMERIC_FEATURES
FUSION_NUMERIC_FEATURES = TABULAR_NUMERIC_FEATURES + VISUAL_NUMERIC_FEATURES

CUSTOMER_CATEGORICAL_FEATURES = ['club_member_status', 'fashion_news_frequency']
ARTICLE_CATEGORICAL_FEATURES = [
    'product_type_name',
    'product_group_name',
    'graphical_appearance_name',
    'colour_group_name',
    'perceived_colour_value_name',
    'perceived_colour_master_name',
    'department_name',
    'index_code',
    'index_name',
    'index_group_name',
    'section_name',
    'garment_group_name',
]
CATEGORICAL_FEATURES = CUSTOMER_CATEGORICAL_FEATURES + ARTICLE_CATEGORICAL_FEATURES
MODEL_NAMES = ('tabular_only', 'image_history', 'late_fusion')

@dataclass(frozen=True)
class V2Defaults:
    n_folds: int = DEFAULT_N_FOLDS
    validation_days: int = DEFAULT_VALIDATION_DAYS
    seed: int = DEFAULT_SEED
    top_k: int = 12
    precision_k: int = 10
    candidate_limit: int = 5000
    visual_neighbors: int = 3000
    co_purchase_per_item: int = 300
    hybrid_weights: tuple[float, ...] = (0.25, 0.45, 0.65)
    negatives_per_positive: int = 1
    train_batch_size: int = 4096
    epochs: int = 3
    learning_rate: float = 1e-3

def ensure_v2_dirs():
    for path in [REPORTS_DIR, REPORTS_DIR / 'folds', REPORTS_DIR / 'gradcam_examples', MODELS_DIR]:
        path.mkdir(parents=True, exist_ok=True)

## Veri yukleme yardimcilari

Bu hucre H&M raw dosyalarini, image klasorunu ve gerekiyorsa embedding cache dosyalarini Kaggle inputlari veya local klasorlerden bulmak icin yardimci fonksiyonlari tanimlar.

In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd



def log(message: str) -> None:
    print(message, flush=True)


def _candidate_raw_dirs() -> list[Path]:
    candidates: list[Path] = []
    env_dir = os.environ.get("HM_RAW_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.append(DATA_DIR / "raw")
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for path in kaggle_root.rglob("transactions_train.csv"):
            candidates.append(path.parent)
    return candidates


def resolve_raw_dir(raw_dir: str | Path | None = None) -> Path:
    candidates = [Path(raw_dir)] if raw_dir else _candidate_raw_dirs()
    for candidate in candidates:
        if (candidate / "transactions_train.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find H&M raw data directory. Set HM_RAW_DIR or pass --raw-dir.")


def resolve_images_dir(raw_dir: Path, images_dir: str | Path | None = None) -> Path:
    if images_dir:
        return Path(images_dir)
    for candidate in [raw_dir / "images", DATA_DIR / "images" / "hm_images"]:
        if candidate.exists():
            return candidate
    return raw_dir / "images"


def _find_file_by_name(root: Path, name: str) -> Path | None:
    if not root.exists():
        return None
    for path in root.rglob(name):
        return path
    return None


def resolve_embedding_paths(
    embeddings_path: str | Path | None = None,
    embedding_ids_path: str | Path | None = None,
) -> tuple[Path, Path]:
    if embeddings_path and embedding_ids_path:
        return Path(embeddings_path), Path(embedding_ids_path)

    env_embeddings = os.environ.get("HM_EMBEDDINGS_PATH")
    env_ids = os.environ.get("HM_EMBEDDING_IDS_PATH")
    if env_embeddings and env_ids:
        return Path(env_embeddings), Path(env_ids)

    local_embeddings = DATA_DIR / "embeddings" / "final_kaggle" / "article_image_embeddings_popular.npy"
    local_ids = DATA_DIR / "embeddings" / "final_kaggle" / "article_image_embedding_ids_popular.csv"
    if local_embeddings.exists() and local_ids.exists():
        return local_embeddings, local_ids

    kaggle_root = Path("/kaggle/input")
    embeddings = _find_file_by_name(kaggle_root, "article_image_embeddings_popular.npy")
    ids = _find_file_by_name(kaggle_root, "article_image_embedding_ids_popular.csv")
    if embeddings and ids:
        return embeddings, ids

    raise FileNotFoundError("Could not find EfficientNet embedding cache paths.")


def read_transactions(raw_dir: Path) -> pd.DataFrame:
    transactions = pd.read_csv(raw_dir / "transactions_train.csv", dtype={"article_id": str})
    transactions["article_id"] = transactions["article_id"].astype(str).str.zfill(10)
    transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])
    transactions["price"] = transactions["price"].astype("float32")
    transactions["sales_channel_id"] = transactions["sales_channel_id"].astype("int8")
    return transactions


def read_customers(raw_dir: Path) -> pd.DataFrame:
    customers = pd.read_csv(raw_dir / "customers.csv")
    customers["FN"] = customers["FN"].fillna(0).astype("float32")
    customers["Active"] = customers["Active"].fillna(0).astype("float32")
    customers["age"] = customers["age"].fillna(customers["age"].median()).astype("float32")
    for column in ["club_member_status", "fashion_news_frequency"]:
        customers[column] = customers[column].fillna("UNKNOWN").astype(str)
    return customers


def read_articles(raw_dir: Path) -> pd.DataFrame:
    articles = pd.read_csv(raw_dir / "articles.csv", dtype={"article_id": str})
    articles["article_id"] = articles["article_id"].astype(str).str.zfill(10)
    for column in articles.columns:
        if articles[column].dtype == "object":
            articles[column] = articles[column].fillna("UNKNOWN").astype(str)
    return articles


def load_core_tables(raw_dir: str | Path | None = None) -> tuple[Path, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    resolved = resolve_raw_dir(raw_dir)
    log(f"Using raw data: {resolved}")
    return resolved, read_transactions(resolved), read_customers(resolved), read_articles(resolved)


def load_embeddings(
    embeddings_path: str | Path | None = None,
    embedding_ids_path: str | Path | None = None,
    mmap_mode: str | None = "r",
) -> tuple[np.ndarray, list[str], dict[str, int]]:
    emb_path, ids_path = resolve_embedding_paths(embeddings_path, embedding_ids_path)
    log(f"Using embeddings: {emb_path}")
    embeddings = np.load(emb_path, mmap_mode=mmap_mode)
    ids_frame = pd.read_csv(ids_path, dtype={"article_id": str})
    article_ids = ids_frame["article_id"].astype(str).str.zfill(10).tolist()
    article_to_index = {article_id: index for index, article_id in enumerate(article_ids)}
    return embeddings, article_ids, article_to_index


def article_image_path(images_dir: Path, article_id: str) -> Path:
    padded = str(article_id).zfill(10)
    nested = images_dir / padded[:3] / f"{padded}.jpg"
    if nested.exists():
        return nested
    return images_dir / f"{padded}.jpg"

## Fold yardimci fonksiyonlari

Bu hucre customer-level split uretimi ve fold leakage kontrolu icin gereken fonksiyonlari tanimlar.

In [ ]:

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold



def build_customer_folds(
    transactions: pd.DataFrame,
    n_folds: int = DEFAULT_N_FOLDS,
    seed: int = DEFAULT_SEED,
    min_history: int = 2,
) -> pd.DataFrame:
    customer_counts = transactions.groupby("customer_id").size()
    eligible_customers = customer_counts[customer_counts >= min_history].index.to_numpy()
    eligible_customers = np.array(sorted(eligible_customers))

    fold_rows: list[dict] = []
    splitter = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold_id, (_, val_idx) in enumerate(splitter.split(eligible_customers)):
        for customer_id in eligible_customers[val_idx]:
            fold_rows.append({"customer_id": customer_id, "fold_id": fold_id})
    return pd.DataFrame(fold_rows)


def validate_fold_protocol(
    transactions: pd.DataFrame,
    folds: pd.DataFrame,
    validation_days: int = DEFAULT_VALIDATION_DAYS,
) -> dict:
    cutoff = transactions["t_dat"].max() - pd.Timedelta(days=validation_days)
    summary = {
        "cutoff": str(cutoff.date()),
        "validation_days": validation_days,
        "folds": [],
    }
    for fold_id in sorted(folds["fold_id"].unique()):
        val_customers = set(folds.loc[folds["fold_id"] == fold_id, "customer_id"])
        train_customers = set(folds.loc[folds["fold_id"] != fold_id, "customer_id"])
        intersection = train_customers & val_customers
        val_tx = transactions[transactions["customer_id"].isin(val_customers)]
        history = val_tx[val_tx["t_dat"] <= cutoff]
        truth = val_tx[val_tx["t_dat"] > cutoff]
        summary["folds"].append(
            {
                "fold_id": int(fold_id),
                "train_customers": len(train_customers),
                "validation_customers": len(val_customers),
                "customer_intersection": len(intersection),
                "validation_history_rows": int(len(history)),
                "validation_truth_rows": int(len(truth)),
                "validation_truth_customers": int(truth["customer_id"].nunique()),
            }
        )
    return summary


def main() -> None:
    parser = argparse.ArgumentParser(description="Create proposal v2 customer-level 5-fold splits.")
    parser.add_argument("--raw-dir", type=Path, default=None)
    parser.add_argument("--n-folds", type=int, default=DEFAULT_N_FOLDS)
    parser.add_argument("--seed", type=int, default=DEFAULT_SEED)
    parser.add_argument("--min-history", type=int, default=2)
    parser.add_argument("--validation-days", type=int, default=DEFAULT_VALIDATION_DAYS)
    parser.add_argument("--output", type=Path, default=REPORTS_DIR / "proposal_v2_fold_splits.csv")
    parser.add_argument("--summary", type=Path, default=REPORTS_DIR / "proposal_v2_fold_protocol_summary.json")
    args = parser.parse_args()

    ensure_v2_dirs()
    _, transactions, _, _ = load_core_tables(args.raw_dir)
    folds = build_customer_folds(transactions, args.n_folds, args.seed, args.min_history)
    summary = validate_fold_protocol(transactions, folds, args.validation_days)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    folds.to_csv(args.output, index=False)
    args.summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    log(f"Saved folds: {args.output}")
    log(f"Saved protocol summary: {args.summary}")
    for row in summary["folds"]:
        if row["customer_intersection"] != 0:
            raise RuntimeError(f"Fold {row['fold_id']} has customer leakage.")

## Parametreler

`N_FOLDS` proposal uyumu icin 5 olarak tutulur. `MIN_HISTORY`, yeterli gecmisi olmayan musterileri elemek icin kullanilir.

In [ ]:
RAW_DIR = None
N_FOLDS = 5
RANDOM_SEED = 42
MIN_HISTORY = 2
VALIDATION_DAYS = 7

## Fold ciktilarini uret

Bu hucre transaction verisini yukler, fold CSV dosyasini yazar, protocol summary JSON dosyasini yazar ve fold kontrol tablosunu gosterir.

In [ ]:
ensure_v2_dirs()
raw_dir, transactions, customers, articles = load_core_tables(RAW_DIR)
folds = build_customer_folds(transactions, N_FOLDS, RANDOM_SEED, MIN_HISTORY)
summary = validate_fold_protocol(transactions, folds, VALIDATION_DAYS)

folds_path = REPORTS_DIR / 'proposal_v2_fold_splits.csv'
summary_path = REPORTS_DIR / 'proposal_v2_fold_protocol_summary.json'
folds.to_csv(folds_path, index=False)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
for row in summary['folds']:
    if row['customer_intersection'] != 0:
        raise RuntimeError(f"Fold {row['fold_id']} has customer leakage.")
print('Saved folds:', folds_path)
print('Saved summary:', summary_path)
display(pd.DataFrame(summary['folds']))